# **Objective**

The goal of this project is to develop a **robust generative search system** capable of accurately interpreting and answering queries derived from a collection of policy documents. Leveraging frameworks such as **LangChain**, the system will integrate document parsing, embedding-based retrieval, and LLM-powered generation to deliver **reliable, context-aware, and grounded responses**.

By the end of this project, we aim to build an end-to-end pipeline that can:

- **Efficiently ingest, preprocess, and index policy documents** of varying formats and structures  
- **Retrieve highly relevant context** using vector-based semantic search  
- **Generate precise, grounded answers** with minimal hallucination  
- **Ensure scalability and modularity**, enabling components such as models, retrievers, or vector databases to be swapped without major refactoring  
- **Provide an extensible architecture** for future enhancements, including agentic workflows, metadata-driven search, and evaluation pipelines


### **Why LangChain?**

For this project, we are using the **LangChain** framework due to its widespread adoption and maturity in building Generative AI applications. LangChain provides a rich set of tools, modular components, and integrations that significantly simplify the development of complex RAG-based systems. With the release of its stable **1.x version**, developers can rely on a consistent API with no major breaking changes expected until the next major release.

Below are the key reasons for choosing LangChain:

- **Modular and Swappable Components:**  
  LangChain allows seamless switching of foundational elements such as LLMs, vector databases, retrievers, or chunking strategies without disrupting the rest of the pipeline. This flexibility makes experimentation and optimization effortless.

- **Strong Community and Ecosystem:**  
  Being one of the earliest and most widely adopted frameworks for LLM applications, LangChain benefits from extensive community support, documentation, and troubleshooting resources.

- **Unified Toolkit for GenAI Development:**  
  LangChain serves as a one-stop solution, providing loaders, text splitters, embedding models, vector store integrations, evaluators, and agentic workflows — all under a unified API.

- **Future-Ready with LangGraph & Tracing:**  
  While this project does not yet integrate tracing or advanced agent workflows, LangChain supports smooth adoption of these features through LangSmith and LangGraph. This makes the ecosystem highly **extensible and future-proof**, enabling easy upgrades without migrating to another framework.


## Installing important libraries

In [5]:
import logging

logging.getLogger().setLevel(logging.WARNING)


In [6]:
%pip install rank-bm25 openai langchain faiss-cpu pypdf tiktoken docarray PyPDF tiktoken langchain-openai flashrank langchain-community pillow sentence-transformers langchain-docling accelerate arize-phoenix
# langchain-docling

Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip is available: 25.2 -> 25.3
[notice] To update, run: python.exe -m pip install --upgrade pip


## Importing important libraries and functionalities 

In [7]:
from dotenv import load_dotenv

import os
import time
from langchain_openai import ChatOpenAI, OpenAIEmbeddings
from langchain_community.document_loaders import PyPDFDirectoryLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_community.vectorstores import FAISS
from langchain.agents.middleware import PIIMiddleware
import re
from typing import List
# from langchain_docling import DoclingLoader


In [8]:
load_dotenv()

True

In [9]:
from langsmith import Client
client = Client()
prompt = client.pull_prompt("insurance-policy-system-prompt:efd03fd9")

## loading and parsing insurance policy PDF documents

In [10]:
from docling.document_converter import DocumentConverter
from langchain_docling import DoclingLoader
from langchain_docling.loader import ExportType
import glob
import pickle
import os

# File to save/load processed documents
DOCS_PICKLE_FILE = "processed_documents.pkl"

# -------------------------------
# 1️⃣ Load documents from pickle if available
# -------------------------------
if os.path.exists(DOCS_PICKLE_FILE):
    with open(DOCS_PICKLE_FILE, "rb") as f:
        documents = pickle.load(f)
    print(f"Loaded {len(documents)} documents from cache.")
else:
    # -------------------------------
    # 2️⃣ Load PDFs (one-time)
    # -------------------------------
    pdf_files = glob.glob("Policy+Documents/*.pdf")
    converter = DocumentConverter()  # no OCR

    documents = []
    for pdf in pdf_files:
        loader = DoclingLoader(
            file_path=pdf,
            converter=converter,
            export_type=ExportType.DOC_CHUNKS
        )
        docs = loader.load()
        documents.extend(docs)

    print(f"Loaded {len(documents)} documents from PDF.")

    # -------------------------------
    # 3️⃣ Save documents to pickle for future runs
    # -------------------------------
    with open(DOCS_PICKLE_FILE, "wb") as f:
        pickle.dump(documents, f)
    print(f"Saved {len(documents)} documents to {DOCS_PICKLE_FILE}.")


Loaded 864 documents from cache.


In [11]:
documents[0].page_content[:100]

"- <<Date>>\n- <<Policyholder's Name>>\n- <<Policyholder's Address>>\n- <<Policyholder's Contact Number>"

## Splitting the documents in chunks for efficient storage and retrival for our vector store

In [12]:
# Split documents into chunks
print("Splitting documents into chunks...")
text_splitter = RecursiveCharacterTextSplitter(
    chunk_size=1000, 
    chunk_overlap=200,
    length_function=len,
    separators=["\n\n", "\n", " ", ""]
)
splits = text_splitter.split_documents(documents)
print(f"✓ Created {len(splits)} document chunks")
print(f"✓ Average chunk size: {sum(len(s.page_content) for s in splits) // len(splits)} characters")

Splitting documents into chunks...
✓ Created 1136 document chunks
✓ Average chunk size: 565 characters


In [13]:
print(splits[0])

page_content='- <<Date>>
- <<Policyholder's Name>>
- <<Policyholder's Address>>
- <<Policyholder's Contact Number>>
Dear <<Policyholder's Name>>,' metadata={'source': 'Policy+Documents\\HDFC-Life-Easy-Health-101N110V03-Policy-Bond-Single-Pay.pdf', 'dl_meta': {'schema_name': 'docling_core.transforms.chunker.DocMeta', 'version': '1.0.0', 'doc_items': [{'self_ref': '#/texts/0', 'parent': {'$ref': '#/groups/0'}, 'children': [], 'content_layer': 'body', 'label': 'list_item', 'prov': [{'page_no': 1, 'bbox': {'l': 72.0, 't': 766.5540649804688, 'r': 115.926, 'b': 758.3620649804687, 'coord_origin': 'BOTTOMLEFT'}, 'charspan': [0, 8]}]}, {'self_ref': '#/texts/1', 'parent': {'$ref': '#/groups/0'}, 'children': [], 'content_layer': 'body', 'label': 'list_item', 'prov': [{'page_no': 1, 'bbox': {'l': 72.0, 't': 755.0340649804688, 'r': 182.527, 'b': 746.8420649804688, 'coord_origin': 'BOTTOMLEFT'}, 'charspan': [0, 23]}]}, {'self_ref': '#/texts/2', 'parent': {'$ref': '#/groups/0'}, 'children': [], 'cont

## Initializing the Embedding model for using it to create embeddings that to be stored in our vectorDB 

In [14]:
# Initialize embeddings model with timeout and retry settings
print("Initializing OpenAI embeddings model...")
embeddings_model = OpenAIEmbeddings(
    model="text-embedding-3-large",  # Using smaller, faster model
    request_timeout=60,  # 60 second timeout
    max_retries=3  # Retry up to 3 times on failure
)
print("✓ Embeddings model initialized")

Initializing OpenAI embeddings model...
✓ Embeddings model initialized


In [15]:
# Test embedding on a single document
print("Testing embeddings on a sample chunk...")
try:
    test_embedding = embeddings_model.embed_documents([splits[0].page_content])
    print(f"✓ Test embedding successful - dimension: {len(test_embedding[0])}")
except Exception as e:
    print(f"✗ Error testing embeddings: {str(e)}")
    raise

Testing embeddings on a sample chunk...


2025-11-27 16:52:35,725 - INFO - HTTP Request: POST https://api.openai.com/v1/embeddings "HTTP/1.1 200 OK"


✓ Test embedding successful - dimension: 3072


#### Initializing cache backed embeddings as it makes our app more efficient by caching the already computed embedding hence saving cost and latency hence improving our application performance leading to rich user experience

In [16]:
from langchain_classic.embeddings import CacheBackedEmbeddings  
from langchain_classic.storage import LocalFileStore 
store = LocalFileStore("./cache/") 

cached_embedder = CacheBackedEmbeddings.from_bytes_store(
    embeddings_model,
    store,
    namespace="semantic-spotter"
)

c:\Users\rocky\AppData\Local\Programs\Python\Python313\Lib\site-packages\langchain_classic\embeddings\cache.py:58: UserWarning: Using default key encoder: SHA-1 is *not* collision-resistant. While acceptable for most cache scenarios, a motivated attacker can craft two different payloads that map to the same cache key. If that risk matters in your environment, supply a stronger encoder (e.g. SHA-256 or BLAKE2) via the `key_encoder` argument. If you change the key encoder, consider also creating a new cache, to avoid (the potential for) collisions with existing keys.
  _warn_about_sha1_encoder()


In [17]:
# Preview first few splits (safe check)
print("Preview of document splits:")
try:
    for i, split in enumerate(splits[:3]):
        print(f"\n--- Split {i+1} ---")
        print(f"Source: {split.metadata.get('source', 'Unknown')}")
        print(f"Page: {split.metadata.get('page', 'Unknown')}")
        print(f"Content preview: {split.page_content[:150]}...")
    print(f"\n✓ Total splits available: {len(splits)}")
except Exception as e:
    print(f"Error previewing splits: {e}")
    raise

Preview of document splits:

--- Split 1 ---
Source: Policy+Documents\HDFC-Life-Easy-Health-101N110V03-Policy-Bond-Single-Pay.pdf
Page: Unknown
Content preview: - <<Date>>
- <<Policyholder's Name>>
- <<Policyholder's Address>>
- <<Policyholder's Contact Number>>
Dear <<Policyholder's Name>>,...

--- Split 2 ---
Source: Policy+Documents\HDFC-Life-Easy-Health-101N110V03-Policy-Bond-Single-Pay.pdf
Page: Unknown
Content preview: Sub: Your Policy no. <<  >>
We are glad to inform you that your proposal has been accepted and the HDFC Life Easy Health ('Policy') being this documen...

--- Split 3 ---
Source: Policy+Documents\HDFC-Life-Easy-Health-101N110V03-Policy-Bond-Single-Pay.pdf
Page: Unknown
Content preview: Policy document:
As an evidence of the insurance contract between HDFC Life Insurance Company Limited and you, the Policy is  enclosed herewith. Pleas...

✓ Total splits available: 1136


#### Computing, storing and loading from/to our vectorDB based on if the embedding are computed and stored already.

In [18]:

def create_vector_store_faiss(splits, embeddings_model, save_path="./faiss_store"):
    """Create and save FAISS vector store."""
    print(f"Creating vector store from {len(splits)} documents...")
    start_time = time.time()
    
    try:
        # Create FAISS store directly from documents
        if os.path.exists(save_path):
            return FAISS.load_local(save_path, embeddings=embeddings_model, allow_dangerous_deserialization=True)
        
        vectordb = FAISS.from_documents(
            documents=splits,
            embedding=embeddings_model
        )
        print(f"✓ FAISS vector store created")
        
        # Save to disk
        os.makedirs(save_path, exist_ok=True)
        vectordb.save_local(save_path)
        print(f"✓ Saved to: {save_path}")
        
        elapsed = time.time() - start_time
        print(f"✓ Time: {elapsed:.1f}s ({elapsed/60:.1f}m)")
        
        return vectordb
    except Exception as e:
        print(f"✗ Error: {type(e).__name__}: {e}")
        raise


In [19]:
# Create the vector store
try:
    vectordb = create_vector_store_faiss(splits, cached_embedder, "./faiss_store")
    print("✓ Vector store ready for similarity search")
except Exception as e:
    print(f"Failed to create vector store: {str(e)}")
    raise


2025-11-27 16:52:35,844 - INFO - Loading faiss with AVX2 support.


Creating vector store from 1136 documents...


2025-11-27 16:52:36,067 - INFO - Successfully loaded faiss with AVX2 support.


✓ Vector store ready for similarity search


#### Hybrid search implemtation 

In [20]:
from langchain_community.retrievers import BM25Retriever
from langchain_classic.retrievers.ensemble import EnsembleRetriever

def create_hybrid_retriever(splits, embeddings_model, vectordb, k_bm25=10, k_vector=10, 
                           weight_bm25=0.4, weight_vector=0.6):
    """Create hybrid retriever combining BM25 keyword search with vector semantic search."""
    print("Creating hybrid retriever...")
    
    # BM25 retriever for keyword matching
    bm25_retriever = BM25Retriever.from_documents(splits)
    bm25_retriever.k = k_bm25
    print(f"✓ BM25 retriever initialized (k={k_bm25})")
    
    # Vector retriever for semantic matching
    vector_retriever = vectordb.as_retriever(search_kwargs={"k": k_vector})
    print(f"✓ Vector retriever initialized (k={k_vector})")
    
    # Ensemble retriever combining both
    hybrid_retriever = EnsembleRetriever(
        retrievers=[bm25_retriever, vector_retriever],
        weights=[weight_bm25, weight_vector]
    )
    print(f"✓ Hybrid retriever created (BM25: {weight_bm25*100}%, Vector: {weight_vector*100}%)")
    
    return hybrid_retriever

Note: Weight tuning does not matter much if we are using cross-encores (Rerankers) otherwise we have to perform weight tuning for optimal retirval  using grid search 

#### Implementing Compression and reranker to improve our retrival relancy/performance further without adding much latency by using lightweight reranker

In [21]:
import traceback
from langchain.tools import tool
from langchain_classic.retrievers.contextual_compression import ContextualCompressionRetriever
from langchain_community.document_compressors import FlashrankRerank

compressor = FlashrankRerank()
compression_retriever = None

if 'vectordb' in globals() and vectordb is not None:

        # Create hybrid retriever
    hybrid_retriever = create_hybrid_retriever(
        splits, 
        cached_embedder, 
        vectordb,
        k_bm25=10,
        k_vector=10,
        weight_bm25=0.4,    # 40% keyword-based
        weight_vector=0.6   # 60% semantic-based
    )
    
    compression_retriever = ContextualCompressionRetriever(
        base_compressor=compressor, base_retriever=hybrid_retriever
    )
    # compression_retriever = ContextualCompressionRetriever(
    #     base_compressor=compressor, base_retriever=vectordb.as_retriever(search_kwargs={"k": 20})
    # )

@tool(response_format="content_and_artifact")
def retrieve_context(query: str):
    """Retrieve information to help answer a query"""


    if 'vectordb' not in globals() and vectordb is None:
        return "No vector store available", []
    retrieved_docs = []
    
    #prefer compression_retriever when available , otherwise use fallback to basic retriever
    try:
        if compression_retriever is not None:
            retrieved_docs = compression_retriever.invoke(
            query
        )
            
        else:
            retrieved_docs = vectordb.as_retriever(search_kwargs={"k": 5}).get_relevant_documents(query)
    # retrieved_docs = vectordb.similarity_search(query, k=2)
    # serialized = "\n\n".join(
    #     (f"Source: {doc.metadata}\n Page Content: {doc.page_content}")
    #     for doc in retrieved_docs
    # )
       
    except Exception as e:
        print("❌ ERROR IN retrieve_context:", e)
        traceback.print_exc()   # 🔥 Shows REAL reason
        return "Retrieval failed", []

    serialized = "\n\n".join(
            f"Source: {d.metadata.get('source','unknown')} | Page: {d.metadata.get('page','?')}\n{d.page_content}"
            for d in retrieved_docs
        )
    return serialized, retrieved_docs

Creating hybrid retriever...
✓ BM25 retriever initialized (k=10)
✓ Vector retriever initialized (k=10)
✓ Hybrid retriever created (BM25: 40.0%, Vector: 60.0%)


In [22]:
# Defensive extraction of a system prompt string
def extract_prompt_text(prompt):
    try:
        # ChatPromptTemplate (LangChain Core)
        msgs = prompt.format_prompt().to_messages()
        if msgs:
            return msgs[0].content
    except Exception:
        pass

    try:
        # Some prompt objects expose a template directly
        return getattr(prompt, "template", None) or getattr(prompt, "prompt", None) or str(prompt)
    except Exception:
        return "You are an insurance policy QA assistant. Use only retrieved documents."

# usage
system_prompt_text = extract_prompt_text(prompt)

#### Finally chaining our retriver tool with LLM and prompt

Note: This our now the go to method for performing RAG instead of LCEL based chaining in Langchain since v1

In [23]:
from langchain.agents import create_agent
from langchain_core.messages import SystemMessage
import logging
logging.getLogger("openai").setLevel(logging.WARNING)
logging.getLogger("langchain").setLevel(logging.WARNING)
# Instantiate the LLM
llm = ChatOpenAI(model_name="gpt-4o-mini", streaming=True)

tools = [retrieve_context]

# prompt = """You are an insurance policy QA assistant. Use ONLY the content returned by the retrieval tool(s) to answer — DO NOT rely on external knowledge or guess.

# REQUIRED BEHAVIOR:
# 1) ALWAYS respond EXACTLY in this two-line format and nothing else:
# answer: [your concise answer here]
# source: [filename.pdf | page X][; filename2.pdf | page Y]  # list one or more sources separated by semicolons

# 2) If the provided documents do NOT contain an answer, respond exactly:
# answer: I don't know — not found in provided documents.
# source: none

# 3) When you can, include a 1-2 sentence quoted excerpt that directly supports your answer. Put the excerpt inside double quotes on the same answer line after the main sentence, followed by the source line.

# 4) Use the retriever tool results only. Do not invent facts, numbers, policy limits, dates, or legal language. If uncertain, say so.

# 5) Format rules:
#    - Numeric amounts: use digits and currency symbol, e.g. $500,000
#    - Dates: YYYY-MM-DD
#    - Keep the answer concise (1-3 short sentences).
#    - Cite page numbers and source filename exactly as returned by the retriever metadata.

# EXAMPLES:
# Correct:
# answer: The policy provides up to $500,000 in life insurance coverage. "Coverage limit: $500,000 (see policy limits section)." 
# source: policy_document_v1.pdf | page 12

# If not found:
# answer: I don't know — not found in provided documents.
# source: none

# If user question is ambiguous:
# answer: Please clarify: do you mean coverage amount or eligibility criteria?
# source: none

# CRITICAL: Use only the retrieved document text. Do not include any additional commentary, operational notes, or tool output. End response after the two required lines."""



agent = create_agent(llm,
                     tools,
                     system_prompt=system_prompt_text,
                     middleware = [# Redact emails in user input before sending to model
                     PIIMiddleware(
                        "email",
                        strategy="redact",
                        apply_to_input=True,
                     ),
                     # Mask credit cards in user input
                     PIIMiddleware(
                        "credit_card",
                        strategy="mask",
                        apply_to_input=True,
                     ),
                     # Block API keys - raise error if detected
                     PIIMiddleware(
                        "api_key",
                        detector=r"sk-[a-zA-Z0-9]{32}",
                        strategy="block",
                        apply_to_input=True,
                     ),])


In [24]:
from openai import OpenAI
openAIClient = OpenAI()
def check_moderation_flag(expression):
    moderation_response = openAIClient.moderations.create(input=expression)
    flagged = moderation_response.results[0].flagged
    return flagged

In [25]:
# ---------- 1) canonicalize text ----------
def canonicalize(text: str) -> str:
    # remove nulls, weird control chars, excessive whitespace, HTML comments, scripts
    text = re.sub(r'<!--.*?-->', ' ', text, flags=re.DOTALL)
    text = re.sub(r'<script.*?>.*?</script>', ' ', text, flags=re.DOTALL|re.IGNORECASE)
    text = text.replace('\x00', ' ')
    text = re.sub(r'[\r\n\t]+', ' ', text)
    text = re.sub(r'\s{2,}', ' ', text)
    return text.strip()

# ---------- 2) basic prompt-injection signature detector (regex) ----------
INJECTION_PATTERNS = [
    r'(?i)ignore (?:previous|earlier) instructions',
    r'(?i)disregard (?:previous|earlier) instructions',
    r'(?i)follow these steps:',
    r'(?i)execute the following',
    r'(?i)now do exactly as follows',
    r'(?i)you are now',
    r'(?i)system message:',
    r'(?i)if you are reading this',
]

def detect_injection(text: str) -> List[str]:
    found = []
    for p in INJECTION_PATTERNS:
        if re.search(p, text):
            found.append(p)
    return found

#### Function that we will be using to answer the user query from our complete RAG system

In [26]:
from langchain.messages import HumanMessage

def insurance_agent(query: str):
    """Runs your LLM agent after sanitization + moderation filtering."""

    query = canonicalize(query)

    injections = detect_injection(query)
    if injections:
        return f"Prompt injection detected! Patterns: {injections}"

    if check_moderation_flag(query):
        return "Input content flagged by moderation filter."

    response = agent.invoke({
        'messages': [HumanMessage(content=query)]
    })

    return response['messages'][-1].content


In [27]:
print(prompt.messages[0].prompt.template)

You are an insurance policy QA assistant. Use ONLY the content returned by the retrieval tool(s) to answer — DO NOT rely on external knowledge or guess.

REQUIRED BEHAVIOR:
1) ALWAYS respond EXACTLY in this two-line format and nothing else:
answer: [your concise answer here]
source: [filename.pdf | page X][; filename2.pdf | page Y]  # list one or more sources separated by semicolons

2) If the provided documents do NOT contain an answer, respond exactly:
answer: I don't know — not found in provided documents.
source: none

3) When you can, include a 1-2 sentence quoted excerpt that directly supports your answer. Put the excerpt inside double quotes on the same answer line after the main sentence, followed by the source line.

4) Use the retriever tool results only. Do not invent facts, numbers, policy limits, dates, or legal language. If uncertain, say so.

5) Format rules:
   - Numeric amounts: use digits and currency symbol, e.g. $500,000
   - Dates: YYYY-MM-DD
   - Keep the answer c

In [28]:
insurance_agent( "Can a 100 year plus person do a term insurance?")

2025-11-27 16:52:41,140 - INFO - HTTP Request: POST https://api.openai.com/v1/moderations "HTTP/1.1 200 OK"
2025-11-27 16:52:43,547 - INFO - HTTP Request: POST https://api.openai.com/v1/chat/completions "HTTP/1.1 200 OK"
2025-11-27 16:52:45,143 - INFO - HTTP Request: POST https://api.openai.com/v1/embeddings "HTTP/1.1 200 OK"
2025-11-27 16:52:49,088 - INFO - HTTP Request: POST https://api.openai.com/v1/chat/completions "HTTP/1.1 200 OK"


"answer: I don't know — not found in provided documents.  \nsource: none"

In [29]:
insurance_agent("what is the Definitions of Critical Illnesses? based on policy?")

2025-11-27 16:52:51,390 - INFO - HTTP Request: POST https://api.openai.com/v1/moderations "HTTP/1.1 200 OK"
2025-11-27 16:52:53,510 - INFO - HTTP Request: POST https://api.openai.com/v1/chat/completions "HTTP/1.1 200 OK"
2025-11-27 16:52:55,443 - INFO - HTTP Request: POST https://api.openai.com/v1/embeddings "HTTP/1.1 200 OK"
2025-11-27 16:52:58,268 - INFO - HTTP Request: POST https://api.openai.com/v1/chat/completions "HTTP/1.1 200 OK"


'answer: The definitions of critical illnesses based on the policy include: Myocardial Infarction (the first heart attack with specific severity) characterized by the death of heart muscle due to inadequate blood supply, supported by symptoms, ECG changes, and enzymatic evidence; and Cancer of specified severity, defined as a malignant tumor with uncontrolled growth and invasion, requiring histological evidence. "The first occurrence of heart attack... must be evidenced by all of the following criteria..." Source: Policy+Documents\\HDFC-Life-Group-Poorna-Suraksha-101N137V02-Policy-Document.pdf | Page: ?; Policy+Documents\\HDFC-Life-Easy-Health-101N110V03-Policy-Bond-Single-Pay.pdf | Page: ?'

In [30]:
insurance_agent("what is the life insurance coverage for disability?")

2025-11-27 16:53:03,675 - INFO - HTTP Request: POST https://api.openai.com/v1/moderations "HTTP/1.1 200 OK"
2025-11-27 16:53:06,317 - INFO - HTTP Request: POST https://api.openai.com/v1/chat/completions "HTTP/1.1 200 OK"
2025-11-27 16:53:06,881 - INFO - HTTP Request: POST https://api.openai.com/v1/embeddings "HTTP/1.1 200 OK"
2025-11-27 16:53:09,811 - INFO - HTTP Request: POST https://api.openai.com/v1/chat/completions "HTTP/1.1 200 OK"


"answer: I don't know — not found in provided documents.  \nsource: none"

In [31]:

insurance_agent( "how to claim insurance?")

2025-11-27 16:53:11,821 - INFO - HTTP Request: POST https://api.openai.com/v1/moderations "HTTP/1.1 200 OK"
2025-11-27 16:53:12,468 - INFO - HTTP Request: POST https://api.openai.com/v1/chat/completions "HTTP/1.1 200 OK"
2025-11-27 16:53:13,042 - INFO - HTTP Request: POST https://api.openai.com/v1/embeddings "HTTP/1.1 200 OK"
2025-11-27 16:53:17,841 - INFO - HTTP Request: POST https://api.openai.com/v1/chat/completions "HTTP/1.1 200 OK"


'answer: To claim insurance, you typically need to provide basic documentation, such as proof of death (if applicable) and a valid discharge receipt. It’s important to notify the insurer as soon as possible; delays may be condoned in genuine cases. "The documents required for processing a claim are: Basic documentation if death is due to Natural Cause."  \nsource: Policy+Documents\\HDFC-Life-Group-Poorna-Suraksha-101N137V02-Policy-Document.pdf | Page: ?'

### Conclusion

The **Semantic Spotter for Insurance Documents** demonstrates an end-to-end approach to building a robust generative search system for highly structured domains. By combining document parsing, vector-based retrieval, and LLM-powered generation, the system achieves:

- **Accurate context retrieval:** Relevant sections of insurance policies are efficiently identified, minimizing irrelevant results.  
- **Scalable RAG pipeline:** Modular components allow swapping models, retrievers, or vector databases without major code changes.  
- **Grounded answer generation:** Responses are based on actual document content, reducing hallucinations and improving reliability.  
- **Future-ready architecture:** LangChain’s ecosystem support, including LangGraph and document compression options, ensures extensibility for more complex agentic workflows.  

Overall, this project validates that a **semantic-aware, modular RAG system** can significantly improve document understanding and query handling in the insurance domain, paving the way for automated assistance, decision support, and enhanced compliance workflows.
